In [1]:
import numpy as np
import pandas as pd
import paretoset
import time
import json
import heapq
import cvxpy as cp
# glpk cvxopt

PATH = '/Users/sasha/Downloads/my_jupyter/tutorial/sb/'
MAX_NUM_SEQ_ALGO = 50
CURR_N_BOXES = 23
MAX_N_BOXES = 26 # int(np.ceil(CURR_N_BOXES * 1.2))
DIM_FACTOR = 165
# AVG_FILL_RATE = 0.6
SLOPE = 0.54
INTERCEPT = 5.1

MAX_BATCH_SIZE = 10

list2str = lambda l: ','.join([str(int(j)) for j in l])
str2list = lambda s: [bool(int(g)) for g in s.split(',')]

def CardStack(df):
    dims = [df['l'].max(), df['w'].max(), df['h'].sum()]
    dims.sort(reverse = True)
    return pd.DataFrame([dims], columns = ['l', 'w', 'h'])

def TwoItem(df):
    a = list(df.iloc[0])
    b = list(df.iloc[1])
    dims = []
    for i in range(3):
        for j in range(3):
            ac = a.copy()
            ae = ac.pop(i)
            bc = b.copy()
            be = bc.pop(j)
            dim = [ae + be, max(ac[0], bc[0]), max(ac[1], bc[1])]
            dim.sort(reverse = True)
            dims.append(dim)
    dims = pd.DataFrame(dims, columns = ['l', 'w', 'h'])
    return dims[paretoset.paretoset(dims, ['min', 'min', 'min'])].reset_index(drop = True)

def TwoItemSeq(df_in, k_max = 9):
    df = df_in.copy()
    df['v'] = df['l'] * df['w'] * df['h']
    df = df.sort_values('v')[['l', 'w', 'h']]
    dims = df.iloc[[0]]
    for r in range(1, len(df.index)):
        # try pairwise addition of the new item to current superitems
        row = df.iloc[[r]]
        dims = pd.concat([TwoItem(pd.concat([row, dims.loc[[rd]]])) for rd in dims.index])
        dims = dims[paretoset.paretoset(dims, ['min', 'min', 'min'])]
        # reduce the number of superitems
        dims['v'] = dims['l'] * dims['w'] * dims['h']
        dims = dims.sort_values('v')[['l', 'w', 'h']].iloc[0:k_max].reset_index(drop = True)
    return dims

def SingleSize(df, is_flat = False):
    if is_flat:
        n = df['q'].iloc[0]
    else:
        n = len(df.index)
    [l, w, h] = [df['l'].iloc[0], df['w'].iloc[0], df['h'].iloc[0]]
    rotations = [[l, w, h], [l, h, w], [w, l, h], [w, h, l], [h, l, w], [h, w, l]]
    dims = []
    for x in range(int(np.ceil(n ** (1/3)))):
        for y in range(x, int(np.ceil((n / (x + 1)) ** (1/2)))):
            z = int(np.ceil(n / (x + 1) / (y + 1)))
            if z >= y + 1:
                for r in rotations:
                    dim = [r[0] * (x + 1), r[1] * (y + 1), r[2] * z]
                    dim.sort(reverse = True)
                    dims.append(dim.copy())
    dims = pd.DataFrame(dims, columns = ['l', 'w', 'h'])
    return dims[paretoset.paretoset(dims, ['min', 'min', 'min'])].reset_index(drop = True)

def DimsToDict(dims):
    dims_dict = []
    for i in dims.index:
        for j in range(dims.loc[i, 'q']):
            dims_dict.append([dims.loc[i, 'p'], 
                              dims.loc[[i],['l', 'w', 'h']].copy(), 
                              dims.loc[i, 'l'] * dims.loc[i, 'w'] * dims.loc[i, 'h']])
    return {k : dims_dict[k] for k in range(len(dims_dict))}

def SizeGroups(df):
    odf = df.copy()
    odf['p'] = (odf['p'] * odf['q']).apply(lambda x: round(x, 2))
    odf = odf.groupby(['l', 'w', 'h'], as_index = False)[['q', 'p']].sum()
    dims_dict = {}
    for i in odf.index:
        dims_i = SingleSize(odf.loc[[i]], True)
        min_vol_i = (dims_i['l'] * dims_i['w'] * dims_i['h']).min()
        dims_dict[i] = [odf.loc[i, 'p'], dims_i, min_vol_i]
    return dims_dict

def EncompassingDims(dims1, dims2, extra = [], size = False):
    if size and 's' not in extra:
        extra.append('s')
    dims = pd.merge(dims1.assign(key = 1), dims2.assign(key = 1), on = 'key').drop('key', axis = 1)
    dims['l'] = dims[['l_x', 'l_y']].max(axis = 1)
    dims['w'] = dims[['w_x', 'w_y']].max(axis = 1)
    dims['h'] = dims[['h_x', 'h_y']].max(axis = 1)
    dims = dims[['l', 'w', 'h'] + extra]
    if not size:
        return dims[paretoset.paretoset(dims[['l', 'w', 'h']], ['min', 'min', 'min'])].reset_index(drop = True)
    else:
        return dims[paretoset.paretoset(dims[['l', 'w', 'h', 's']], ['min', 'min', 'min', 'min'])].reset_index(drop = True)

def FullyEncompassingDims(dims_dict):
    dims = pd.DataFrame([[0, 0, 0]], columns = ['l', 'w', 'h'])
    for key in dims_dict:
        dims = EncompassingDims(dims, dims_dict[key][1])
    return dims

def CrossTwoItem(dims1, dims2):
    dims = []
    columns = ['l', 'w', 'h']
    for i in dims1.index:
        for j in dims2.index:
            dims.append(TwoItem(pd.concat([dims1.loc[[i], columns], dims2.loc[[j], columns]], axis = 0)))
    dims = pd.concat(dims, axis = 0)
    return dims[paretoset.paretoset(dims, ['min', 'min', 'min'])].reset_index(drop = True)

def MergeSmallest(dims_dict, max_d = None):
    sort_vol = pd.DataFrame([[key, dims_dict[key][2]] for key in dims_dict], columns = ['i', 'v']).sort_values('v')
    new_index = sort_vol['i'].max() + 1
    obj1 = dims_dict.pop(sort_vol.iloc[0,0])
    obj2 = dims_dict.pop(sort_vol.iloc[1,0])
    dims = CrossTwoItem(obj1[1], obj2[1])
    dims['v'] = dims[['l', 'w', 'h']].prod(axis = 1)
    min_vol = dims['v'].min()
    if max_d:
        dims = dims.sort_values('v').iloc[:max_d]
    dims_dict[new_index] = [obj1[0] + obj2[0], dims[['l', 'w', 'h']], min_vol]
    return dims_dict

def MergeSmallestSeq(dims_dict, max_d = None):
    all_dims = []
    max_shipments = len(dims_dict)
    for i in range(max_shipments):
        dims = FullyEncompassingDims(dims_dict)
        dims['s'] = len(dims_dict)
        dims['p'] = json.dumps([round(dims_dict[k][0], 2) for k in dims_dict])
        all_dims.append(dims)
        if len(dims_dict) > 1:
            dims_dict = MergeSmallest(dims_dict, max_d)
    all_dims = pd.concat(all_dims, axis = 0)
    return all_dims[paretoset.paretoset(all_dims[['l', 'w', 'h', 's']], ['min', 'min', 'min', 'min'])].reset_index(drop = True)
    
def CheckFit(dims, boxes, many_to_1 = False):
    if many_to_1:
        group_cols = ['n', 'p']
    else:
        group_cols = ['n']
        
    fit = pd.merge(boxes.assign(key = 1), dims.assign(key = 1), on = 'key').drop('key', axis = 1)
    fit['f'] = (fit['L'] >= fit['l']) & (fit['W'] >= fit['w']) & (fit['H'] >= fit['h'])
    fit = fit[group_cols + ['f']].groupby(group_cols).max().reset_index()
    return fit

def MinMaxSum(lst, k):
    vals = np.sort(np.array(lst))[::-1]
    idxs = np.argsort(np.array(lst))[::-1]
    subsets = [(0, []) for _ in range(k)]
    heapq.heapify(subsets)
    for i in range(len(lst)):
        m = heapq.heappop(subsets)
        heapq.heappush(subsets, (m[0] + vals[i], m[1] + [idxs[i]]))
    return [(s, i) for s in range(k) for i in subsets[s][1]]

def CardStackMulti(df, max_box = None):
    l = df['l'].max()
    w = df['w'].max()
    hdf = pd.concat([pd.concat([df[df['q'] == q][['h', 'p']]] * q) for q in df['q'].drop_duplicates()]).reset_index(drop = True)
    dims = []
    if not max_box:
        max_box = df['q'].sum()
    for s in range(max_box):
        group_df = pd.DataFrame(MinMaxSum(list(hdf['h']), s + 1), columns = ['group', 'index']).set_index('index')
        hdf['group'] = group_df['group']
        h = hdf.groupby('group')['h'].sum().max()
        dim = [l, w, h]
        dim.sort(reverse = True)
        dims.append(dim + [s + 1, json.dumps(list(hdf.groupby('group')['p'].sum().apply(lambda x: round(x, 2))))])
    dims = pd.DataFrame(dims, columns = ['l', 'w', 'h', 's', 'p'])
    return dims[paretoset.paretoset(dims[['l', 'w', 'h', 's']], ['min', 'min', 'min', 'min'])].reset_index(drop = True)

def Flatten(df):
    odf = pd.concat([pd.concat([df[df['q'] == q]] * q) for q in df['q'].drop_duplicates()]).reset_index(drop = True)
    odf['q'] = 1
    return odf

def FullSearch(df, is_dict = False, multi_box = True):
    if not is_dict:
        n = len(df.index)
    else:
        n = len(df)
    masks = []
    lengths = []
    for i in range(1, 2 ** n):
        mask = [bool(int(b)) for b in list(str(bin(i))[2:])]
        mask = [False] * (n - len(mask)) + mask
        lengths.append(sum(mask))
        masks.append(mask)
    sort_masks = sorted(masks, key = sum)

    mask_dims = {}
    if multi_box:
        mult_dims = {}
    for m in sort_masks:
        k = sum(m)
        if k == 1:
            if not is_dict:
                item = df[pd.Series(m)][['l', 'w', 'h', 'p']].copy()
                mask_dims[list2str(m)] = [item[['l', 'w', 'h']], item['p'].sum()]
            else:
                item_id = sum([i * m[i] for i in range(n)])
                item = df[item_id][1].copy()
                item['p'] = df[item_id][0]
                mask_dims[list2str(m)] = [item[['l', 'w', 'h']], df[item_id][0]]
            if multi_box:
                item['s'] = 1
                item['p'] = item['p'].apply(lambda x: json.dumps([x]))
                mult_dims[list2str(m)] = item[['l', 'w', 'h', 's', 'p']]
        else:
            sub_masks = [sm[-k:] for sm in masks[:2 ** (k - 1) - 1]]
            active_pos = [i for i in range(n) if m[i]]
            dims = []
            mdims = []
            for sm in sub_masks:
                m1 = m.copy()
                m2 = m.copy()
                for pos in range(k):
                    m1[active_pos[pos]] = sm[pos]
                    m2[active_pos[pos]] = not sm[pos]
                [dims1, p1] = mask_dims[list2str(m1)]
                [dims2, p2] = mask_dims[list2str(m2)]
                dims.append(CrossTwoItem(dims1, dims2))
                if multi_box:
                    # multi-package dims are part 1 as a whole plus part 2 as partition
                    mdim2 = mult_dims[list2str(m2)]
                    mdim = EncompassingDims(dims1, mdim2, extra = ['p', 's'], size = True)
                    mdim['s'] = mdim['s'] + 1
                    mdim['p'] = mdim['p'].apply(lambda x: json.dumps([round(float(p1), 2)] + json.loads(x)))
                    mdims.append(mdim)
            dims = pd.concat(dims)
            dims = dims[paretoset.paretoset(dims, ['min', 'min', 'min'])].reset_index(drop = True)
            mask_dims[list2str(m)] = [dims, p1 + p2]
            if multi_box:
                # 2+package dims and 1-package dims
                mdim = dims.copy()
                mdim['s'] = 1
                mdim['p'] = json.dumps([round(float(p1 + p2), 2)])
                mdims.append(mdim)
                mdims = pd.concat(mdims)
                mdims = mdims[paretoset.paretoset(mdims[['l', 'w', 'h', 's']], ['min', 'min', 'min', 'min'])].reset_index(drop = True)
                mult_dims[list2str(m)] = mdims[['l', 'w', 'h', 's', 'p']]
    if multi_box:
        return mult_dims[list2str(sort_masks[-1])]
    out_dims = mask_dims[list2str(sort_masks[-1])][0]
    out_dims['s'] = 1
    out_dims['p'] = json.dumps([mask_dims[list2str(sort_masks[-1])][1]])
    return out_dims

def BoxMatch(actual, catalog, cols_match, act_id, tie_break_cols = [], p = 2):
    match = actual.merge(catalog, how = 'cross')
    match['diff'] = match.apply(lambda x: sum([abs(x[c_a] - x[cols_match[c_a]]) ** p for c_a in cols_match]) ** (1 / p), axis = 1)
    match = match.sort_values([act_id, 'diff'] + tie_break_cols).groupby(act_id).first().reset_index()
    return match



/Users/sasha/opt/anaconda3/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [5]:
def InferTranCost(df):
    es = df.copy() 
    
    # if df['T'].notna().sum() > 1000...

    is_box = pd.concat([df[['n']], df[['L', 'W', 'H']].apply(lambda x: 
        'x'.join([str(c) for c in x]), axis = 1)], axis = 1).drop_duplicates().groupby('n').count()
    act_box_data = df[['n', 'L', 'W', 'H']].drop_duplicates().merge(is_box[is_box[0] == 1].reset_index()[['n']], on = 'n', how = 'right')
    match = BoxMatch(act_box_data, ul_bx, {'L_x' : 'L_y', 'W_x' : 'W_y', 'H_x' : 'H_y'}, 'n_x', ['c'], 2)
    es = es.merge(match[['n_x', 'c']], left_on = 'n', right_on = 'n_x', how = 'left')
    es['Box Cost'] = es['c'].fillna(0)

    es['W_s'] = es['P']
    es['W_i'] = es['p'] * es['q']
    es['V_i'] = 0 #es['l'] * es['w'] * es['h'] * es['q'] / AVG_FILL_RATE
    es['V_s'] = es['L'] * es['W'] * es['H']

    es['Shipment Cost'] = es['T']#0

    es_agg = es[['o', 'Shipment Cost', 'Box Cost', 'W_s', 'V_s', 'W_i', 'V_i']].fillna(0)
    es_agg = es_agg.groupby(['o', 'Shipment Cost', 'W_s', 'V_s'], as_index = False).sum()
    es_agg[['W_s', 'V_s', 'W_i', 'V_i']] = es_agg[['W_s', 'V_s', 'W_i', 'V_i']].fillna(0)
    es_agg['BW'] = es_agg.apply(lambda x: np.ceil(max(x['W_s'], x['W_i'],x['V_i']/DIM_FACTOR, (x['V_s'] if x['V_s'] else x['V_i']) / DIM_FACTOR)), axis = 1)

    # es_agg['Shipment Cost'] = 6.0 + 0.3 * es_agg['BW']

    t = es_agg[es_agg['Shipment Cost'] > 0][['o', 'Shipment Cost', 'Box Cost', 'BW', 'V_i', 'V_s']]
    # display(t)

    y = 'Shipment Cost'
    x = 'BW'
    n = len(t.index)

    x_m = t[x].mean()
    y_m = t[y].mean()

    SLOPE = ((t[x] - x_m) * (t[y] - y_m)).sum() / ((t[x] - x_m) * (t[x] - x_m)).sum()
    INTERCEPT = y_m - SLOPE * x_m

    return [SLOPE, INTERCEPT, t, list(match['n_y'])]

def ComputeCost(odf, bx, o):
    num_items = odf['q'].sum()
    num_sizes = len(odf[['l', 'w', 'h']].drop_duplicates().index)
    
    dims = []
    dims.append(CardStackMulti(odf, 10))

    if num_items < 5:
        dims.append(FullSearch(Flatten(odf)))
    else:
        if num_items < 10:
            dims.append(MergeSmallestSeq(DimsToDict(odf), max_d = 10))
        if num_sizes < min(4, num_items):
            dims.append(FullSearch(SizeGroups(odf), is_dict = True))
        if 4 <= num_sizes < min(10, num_items): 
            dims.append(MergeSmallestSeq(SizeGroups(odf), max_d = 10))


    dims = pd.concat(dims, axis = 0)
    dims = dims[paretoset.paretoset(dims[['l', 'w', 'h', 's']], ['min', 'min', 'min', 'min'])].reset_index(drop = True)


    cost = CheckFit(dims, bx[['n', 'L', 'W', 'H']], many_to_1 = True).merge(bx, on = 'n')
    cost['V'] = cost['L'] * cost['W'] * cost['H']
    cost['o'] = o
    cost['t'] = cost.apply(lambda x: sum([FULFILLMENT_COST_P(x, p) for p in json.loads(x['p'])]), 
                           axis = 1) * cost['f'].apply(lambda x: 1 if x else None)
    cost['p'] = cost.apply(lambda x: x['p'] if x['f'] else None, axis = 1)
    cost = cost[['o', 'n', 't', 'p']].sort_values('t').groupby(['o', 'n'], as_index = False).first()
    return cost

def RunCartonization(in_df, ul_bx):
    df = in_df.copy()
    df = df.groupby(['o', 'l', 'w', 'h', 'p'], as_index = False).sum()
    df['key'] = df.apply(lambda x: '_'.join([str(round(x[c], 2)) for c in ['l', 'w', 'h', 'p', 'q']]), axis = 1)
    order_keys = df[['o', 'key']].sort_values('key').groupby('o', as_index = False).agg(lambda x: '+'.join(x))
    unique_orders = order_keys.groupby('key').first().reset_index(drop = True)

    df = in_df.merge(unique_orders, on = 'o', how = 'right')
    orders = df['o'].drop_duplicates()
    df = df.set_index('o')

    bx = ul_bx.copy()

    start = time.time()
    iterator = 0
    costs = []
    pounds = []
    print('unique orders', len(orders))
    for o in orders:
        odf = df.loc[[o]].reset_index(drop = True)

        num_items = odf['q'].sum()
        num_sizes = len(odf[['l', 'w', 'h']].drop_duplicates().index)

        na_flag = max(odf['l'].isna().sum(), odf['l'].isna().sum())
        if na_flag == 0:
            if num_items > MAX_BATCH_SIZE:
                batch_num = int(np.ceil(num_items / MAX_BATCH_SIZE))
                batch_size = int(np.ceil(num_items / batch_num))
                bodf = Flatten(odf).sort_values(['l', 'w', 'h'])
                cost_dfs = []
                for b in range(batch_num):
                    cost_dfs.append(ComputeCost(bodf.iloc[b * batch_size : (b + 1) * batch_size], bx, o))
                cost = pd.concat([cdf.set_index(['o', 'n']) for cdf in cost_dfs], axis = 1)
                cost['tt'] = cost[['t']].sum(axis = 1)
                cost['pt'] = cost[['p']].apply(lambda x: json.dumps(sum([[round(e, 2) for e in json.loads(y)] for y in x if y is not None], [])), axis = 1)
                cost['isna'] = cost[['t']].isna().any(axis = 1)
                cost = cost[['tt', 'pt', 'isna']].reset_index()
                cost['t'] = cost['tt'] * cost['isna'].apply(lambda x: 1 if not x else None)
                cost['p'] = cost.apply(lambda x: x['pt'] if not x['isna'] else None, axis = 1)
            else:
                cost = ComputeCost(odf, bx, o)

            costs.append(cost[['o', 'n', 't']].pivot(index = 'o', columns = 'n', values = 't'))
            pounds.append(cost[['o', 'n', 'p']].pivot(index = 'o', columns = 'n', values = 'p'))
            iterator += 1
            if iterator % 100 == 0:
                print(iterator, time.time() - start)
            #if iterator == 100:
            #    break

    print(time.time() - start)
    costs = pd.concat(costs, axis = 0)
    pounds = pd.concat(pounds, axis = 0)
    costs = costs.merge(order_keys, on = 'o', how = 'inner').drop(columns = ['o']).merge(order_keys, on = 'key', how = 'inner').drop(columns = 'key')
    pounds = pounds.merge(order_keys, on = 'o', how = 'inner').drop(columns = ['o']).merge(order_keys, on = 'key', how = 'inner').drop(columns = 'key')
    return [costs.set_index('o'), pounds.set_index('o')]

def BreakCosts(costs, pounds, ul_bx):
    box_costs = pounds.applymap(lambda x: len(json.loads(x)) if x else None)
    ul_bc = ul_bx[['n', 'c']].set_index('n')
    for col in box_costs:
        box_costs[col] *= ul_bc.loc[col, 'c']
    tran_costs = costs - box_costs
    return [box_costs, tran_costs]

def GreedySearch(ul_bx, costs, tran_costs):

    names = list(ul_bx['n'])

    current = [costs.count().sort_values(ascending = False).index[0]]
    min_cost = costs[current].min(axis = 1).sum()
    min_tran_cost = tran_costs[current].min(axis = 1).sum()

    output = [[1, round(min_cost, 2), round(min_tran_cost, 2), current[0]]]

    for i in range(1, MAX_N_BOXES):
        min_name = ''
        for n in names:
            c = costs[current + [n]].min(axis = 1).sum()
            if c < min_cost:
                min_cost = c
                min_tran_cost = tran_costs[current + [n]].min(axis = 1).sum()
                min_name = n
        if min_name != '':
            output.append([1 + i, round(min_cost, 2), round(min_tran_cost, 2), min_name])
            current += [min_name]
        else:
            break

    greedy_search = pd.DataFrame(output, columns = ['box_num', 'cumulative_cost', 'transport_cost', 'incremental_box'])
    greedy_search = greedy_search.merge(ul_bx[['n', 'L', 'W', 'H', 'c']], left_on = 'incremental_box', right_on = 'n', how = 'inner')
    greedy_search = greedy_search.drop(columns = ['n']).set_index('box_num')

    return [greedy_search, output]

def BreakevenPoint(df, costs, t, greedy_search):
    ord_with_cost = list(set(t['o']).intersection(set(costs.index)).intersection(set(df[df['n'] != 'none']['o'])))
    h_c = t.set_index('o').loc[ord_with_cost][['Shipment Cost']].sum().sum()
    h_b = t.set_index('o').loc[ord_with_cost][['Box Cost']].sum().sum()
    h_t = t.set_index('o').loc[ord_with_cost][['Shipment Cost', 'Box Cost']].sum().sum()

    for i in range(len(greedy_search.index)):
        parity_set = list(greedy_search['incremental_box'].iloc[:i + 1])
        c = tran_costs.loc[ord_with_cost][parity_set].min(axis = 1).sum()
        if c < h_c:
            break
    parity_count = i + 1
    return [parity_count, h_c, h_b, h_t]

def BoxAssortmentOptimization(costs, suite_size, is_integer = False, verbose = False):
    C = np.array(costs)
    [n, m] = [C.shape[0], C.shape[1]]
    if is_integer:
        r = cp.Variable(m, boolean = True)
    else:
        r = cp.Variable(m, nonneg = True)
    x = cp.Variable((n, m), nonneg = True)
    constraints = [x @ np.ones(m) == 1, x <= C, cp.sum(r) <= suite_size] + [x[:, i] <= r[i] for i in range(m)] 
    if not is_integer:
        constraints.append(r <= 1)

    prob = cp.Problem(cp.Minimize(cp.sum(cp.multiply(C, x))), constraints)
    
    if is_integer:
        prob.solve(solver = 'GLPK_MI', verbose = verbose, max_iters = 1000, abstol = 10 ** (-3), reltol = 10 ** (-3))
    else:
        prob.solve(solver = 'ECOS', verbose = verbose, max_iters = 1000, abstol = 10 ** (-3), reltol = 10 ** (-3))
    
    return [round(prob.value, 3), r.value]

def RunOptimization(costs, suite_sizes):

    eff_bx = list(costs.reset_index().melt(id_vars = ['o'], 
      var_name = ['n'], value_name = 'c_s').sort_values('c_s').groupby('o').first()['n'].drop_duplicates())

    sm = costs[eff_bx].fillna(0)
    sm = sm[sm.sum(axis = 1) > 0]
    sm['all'] = sm.agg(list, axis = 1).apply(json.dumps)
    sm = sm.groupby('all', as_index = False).sum().drop(columns = 'all')
    sm = sm[sm.sum(axis = 1) > 0]

    bao = {}

    start = time.time()
    for suite_size in suite_sizes:

        [obj, rv] = BoxAssortmentOptimization(sm, suite_size, is_integer = False)

        frac_ba = [eff_bx[i] for i in range(len(rv)) if rv[i] > 10 ** (-3)]
        if len(frac_ba) > suite_size:
            ism = sm[frac_ba]
            [iobj, irv] = BoxAssortmentOptimization(ism, suite_size, is_integer = True)
            iba = [frac_ba[i] for i in range(len(irv)) if irv[i] > 0.5]
        else:
            iobj = obj
            iba = frac_ba
        bao[suite_size] = (obj, iobj, iba)
        print(suite_size, time.time() - start)

    return bao

def Utilization(costs, pounds, boxes, bx):
    choices = costs[boxes].reset_index().melt(id_vars = ['o'], var_name = 'n', value_name = 't').sort_values('t').groupby('o', as_index = False).first()
    pct_unfit = round(choices[choices['t'].isna()].shape[0] / choices.shape[0], 4)
    choices = choices.merge(pounds[boxes].reset_index().melt(id_vars = ['o'], var_name = 'n', value_name = 'p'), on = ['o', 'n'], how = 'left')
    choices['s'] = choices['p'].apply(lambda x: len(json.loads(x)) if x is not None else None)
    choices = choices.groupby('n', as_index = False).sum().sort_values('s', ascending = False)
    choices = choices.merge(bx[['n', 'L', 'W', 'H', 'c']], on = 'n', how = 'left')
    choices['Dims'] = choices.apply(lambda x: (str(x['L']) + 'x' + str(x['W']) + 'x' + str(x['H']) + 'x').replace('.0x', 'x')[:-1], axis = 1)
    choices['Cumulative Utilization Pct'] = (choices['s'].cumsum() / (1+choices['s'].sum())).apply(lambda x: str(round(x * 100, 1)) + '%')
    choices = choices[['n', 's', 'Cumulative Utilization Pct', 'Dims', 't', 'L', 'W', 'H', 'c']].rename(columns = {'s' : 'Total Utilization', 'n' : 'Box Name'})
    return [pct_unfit, choices]

In [3]:
ul_bx = pd.read_csv(PATH + 'arka_boxes.csv')
act_box_data = pd.read_csv(PATH + 'bx.csv')
match = BoxMatch(act_box_data, ul_bx, {'L_x' : 'L_y', 'W_x' : 'W_y', 'H_x' : 'H_y'}, 'n_x', ['c'], 2)
baseline_boxes = list(match['n_y'])
print(baseline_boxes)
match

['S-4951', 'S-4851', 'S-4125', 'S-4127', 'S-4487', 'S-4488', 'S-4838', 'S-16738', 'S-4695', 'S-21600', 'S-23974', 'S-4626', 'S-18388', 'S-4792', 'S-4201', 'S-4507', 'S-21037', 'S-4452', 'S-17999', 'S-4050', 'S-4512', 'S-4307', 'S-4515']


,n_x,Unnamed: 0,L_x,W_x,H_x,n_y,L_y,W_y,H_y,P,c,diff
0,Box-10x6x5,0,10,6,5,S-4951,10.0,6.0,5.0,0,0.90,0.000000
1,Box-10x8x3,1,10,8,3,S-4851,10.0,8.0,3.0,0,0.86,0.000000
2,Box-12x12x12,2,12,12,12,S-4125,12.0,12.0,12.0,0,1.03,0.000000
3,Box-12x6x4,3,12,6,4,S-4127,12.0,6.0,4.0,0,0.59,0.000000
4,Box-12x9x5,4,12,9,5,S-4487,12.0,9.0,5.0,0,1.17,0.000000
5,Box-12x9x7,5,12,9,7,S-4488,12.0,9.0,7.0,0,1.36,0.000000
6,Box-13x10x8,6,13,10,8,S-4838,13.0,10.0,8.0,0,1.49,0.000000
7,Box-14x12x3,7,14,12,3,S-16738,14.0,12.0,3.0,0,1.38,0.000000
8,Box-14x12x4,8,14,12,4,S-4695,14.0,12.0,4.0,0,1.39,0.000000
9,Box-15x11x4,9,15,11,4,S-21600,15.0,11.0,4.0,0,1.44,0.000000


In [9]:
ul_bx = pd.read_csv(PATH + 'arka_boxes.csv')

df = pd.read_csv(PATH + 'data.csv')
es_df = df[['o', 'l', 'w', 'h', 'q', 'p']].copy()

#[SLOPE, INTERCEPT, t, baseline_ba] = InferTranCost(df)
#SLOPE = 0.3
#INTERCEPT = 6

FULFILLMENT_COST = lambda x: x['c'] + INTERCEPT + SLOPE * np.ceil(max(x['p'] + x['P'], x['V'] / DIM_FACTOR))
FULFILLMENT_COST_P = lambda x, p: x['c'] + INTERCEPT + SLOPE * np.ceil(max(p + x['P'], x['V'] / DIM_FACTOR))

[costs, pounds] = RunCartonization(es_df, ul_bx)
costs.to_csv(PATH + 'costs.csv')
pounds.to_csv(PATH + 'pounds.csv')
[box_costs, tran_costs] = BreakCosts(costs, pounds, ul_bx)

[greedy_search, output] = GreedySearch(ul_bx, costs, tran_costs)
greedy_search.to_csv(PATH + 'greedy_search.csv')

#[parity_count, h_c, h_b, h_t] = BreakevenPoint(df, costs, t, greedy_search)

suite_sizes = [20, CURR_N_BOXES, MAX_N_BOXES] #, int((MAX_N_BOXES + CURR_N_BOXES) / 2)

bao = RunOptimization(costs, suite_sizes)
print(bao)

utilizations = []
print('size, transp, box, total, volume, pct unfit')
print('baseline')
#print(CURR_N_BOXES, round(h_c), round(h_b), round(h_t), round(t[['V_i', 'V_s']].max(axis = 1).sum() / 1728))
[pct_unfit,utilization] = Utilization(costs, pounds, baseline_boxes, ul_bx)
utilizations.append(utilization)
tot_cost = round(utilization['t'].sum())
box_cost = round((utilization['Total Utilization'] * utilization['c']).sum())
box_volume = round((utilization['Total Utilization'] * utilization['L'] * utilization['W'] * utilization['H']).sum() / 1728)
print('original 23', tot_cost - box_cost, box_cost, tot_cost, box_volume, pct_unfit)
print('optimized')
#ord_with_cost = list(set(t['o']).intersection(set(costs.index)).intersection(set(df[df['n'] != 'none']['o'])))
for size in suite_sizes:
    #[pct_unfit,utilization] = Utilization(costs, pounds, bao[size][2], ul_bx)
    [pct_unfit,utilization] = Utilization(costs, pounds, list(greedy_search.iloc[0:size]['incremental_box']), ul_bx)
    
    utilizations.append(utilization)
    tot_cost = round(utilization['t'].sum())
    box_cost = round((utilization['Total Utilization'] * utilization['c']).sum())
    box_volume = round((utilization['Total Utilization'] * utilization['L'] * utilization['W'] * utilization['H']).sum() / 1728)
    print(size, tot_cost - box_cost, box_cost, tot_cost, box_volume, pct_unfit)

for i in range(len(utilizations)):
    utilizations[i].to_csv(PATH + 'utilization_' + str(i) + '.csv')

unique orders 7243
100 61.97132587432861
200 111.2872679233551
300 157.63926100730896
400 210.13334894180298
500 271.7478358745575
600 346.42921710014343
700 401.2949249744415
800 453.052227973938
900 511.53954315185547
1000 572.1816170215607
1100 627.9253468513489
1200 664.4783990383148
1300 731.6963589191437
1400 764.8648760318756
1500 800.0471487045288
1600 836.0186228752136
1700 867.3872940540314
1800 902.5031597614288
1900 938.9931080341339
2000 977.5393569469452
2100 1025.900444984436
2200 1057.752739906311
2300 1108.0347919464111
2400 1145.055510044098
2500 1177.5513789653778
2600 1224.6794669628143
2700 1283.100921869278
2800 1344.5732288360596
2900 1408.6200079917908
3000 1469.875925064087
3100 1526.535945892334
3200 1560.476791858673
3300 1611.825779914856
3400 1666.7834198474884
3500 1716.8898351192474
3600 1767.0053119659424
3700 1819.0050010681152
3800 1881.719302892685
3900 1928.7921857833862
4000 1967.7372810840607
4100 2012.4238150119781
4200 2053.5645949840546
4300 210

In [10]:
utilizations = []
print('size, transp, box, total, volume, pct unfit')
print('baseline')
#print(CURR_N_BOXES, round(h_c), round(h_b), round(h_t), round(t[['V_i', 'V_s']].max(axis = 1).sum() / 1728))
[pct_unfit,utilization] = Utilization(costs, pounds, baseline_boxes, ul_bx)
utilizations.append(utilization)
tot_cost = round(utilization['t'].sum())
box_cost = round((utilization['Total Utilization'] * utilization['c']).sum())
box_volume = round((utilization['Total Utilization'] * utilization['L'] * utilization['W'] * utilization['H']).sum() / 1728)
print('original 23', tot_cost - box_cost, box_cost, tot_cost, box_volume, pct_unfit)
print('optimized')
#ord_with_cost = list(set(t['o']).intersection(set(costs.index)).intersection(set(df[df['n'] != 'none']['o'])))
for size in suite_sizes:
    [pct_unfit,utilization] = Utilization(costs, pounds, bao[size][2], ul_bx)
    utilizations.append(utilization)
    tot_cost = round(utilization['t'].sum())
    box_cost = round((utilization['Total Utilization'] * utilization['c']).sum())
    box_volume = round((utilization['Total Utilization'] * utilization['L'] * utilization['W'] * utilization['H']).sum() / 1728)
    print(size, tot_cost - box_cost, box_cost, tot_cost, box_volume, pct_unfit)

for i in range(len(utilizations)):
    utilizations[i].to_csv(PATH + 'utilization' + str(i) + '.csv')

size, transp, box, total, volume, pct unfit
baseline
original 23 82912 11626 94538 5838 0.0006
optimized
20 76036 8982 85018 4831 0.0
23 75493 8920 84413 4736 0.0
26 75180 8777 83957 4685 0.0


In [12]:
len(baseline_boxes)

23

In [4]:


utilizations[0].to_csv(PATH + 'utilization.csv')


In [11]:
bao

{13: (450664.423,
  450992.21,
  ['S-4120',
   'S-4121',
   'S-4312',
   'S-12772',
   'S-4160',
   'S-4215',
   'S-4969',
   'S-4183',
   'S-4471',
   'S-4659',
   'S-22698',
   'S-4500',
   'S-13288']),
 16: (445081.3,
  445566.96,
  ['S-4363',
   'S-4120',
   'S-4119',
   'S-4346',
   'S-4121',
   'S-4559',
   'S-12607',
   'S-12772',
   'S-4160',
   'S-4969',
   'S-4183',
   'S-4471',
   'S-4454',
   'S-22698',
   'S-4500',
   'S-13288']),
 20: (440534.818,
  440735.76,
  ['S-4363',
   'S-4120',
   'S-4119',
   'S-4346',
   'S-4121',
   'S-4559',
   'S-12607',
   'S-12772',
   'S-4125',
   'S-4160',
   'S-4969',
   'S-4183',
   'S-4397',
   'S-4399',
   'S-4219',
   'S-4454',
   'S-4663',
   'S-22698',
   'S-4500',
   'S-13288'])}

In [12]:
"""
size, transp, box, total, volume, pct unfit
baseline
original 16 433837 70697 504534 43879 0.0045
optimized
13 397194 53798 450992 33534 0.0001
16 392522 53045 445567 32952 0.0001
20 388878 51858 440736 31990 0.0001
"""

'\nsize, transp, box, total, volume, pct unfit\nbaseline\noriginal 16 433837 70697 504534 43879 0.0045\noptimized\n13 397194 53798 450992 33534 0.0001\n16 392522 53045 445567 32952 0.0001\n20 388878 51858 440736 31990 0.0001\n'

In [14]:
greedy_search

,cumulative_cost,transport_cost,incremental_box,L,W,H,c
box_num,,,,,,,
1,1946807.28,1550243.1,S-4848,48.00,24.00,12.00,9.21
2,685920.52,585922.5,S-4160,16.00,12.00,10.00,1.34
3,591281.41,512888.7,S-22698,26.00,18.00,14.00,4.47
4,514374.16,449236.2,S-4120,12.00,10.00,8.00,0.91
5,494369.48,433958.7,S-4399,18.00,18.00,12.00,2.21
6,477676.60,419023.8,S-4312,12.00,9.00,3.00,0.77
7,471378.34,413264.1,S-4215,12.00,12.00,4.00,0.89
8,465837.62,408309.9,S-12772,40.00,18.00,8.00,5.46
9,461074.63,405431.4,S-4121,12.00,8.00,6.00,0.68


In [18]:
greedy_search.to_csv(PATH + 'greedy_search.csv')

In [19]:
output

[[1, 7089261.81, 5613331.32, 'S-19870'],
 [2, 3096895.44, 2480999.21, 'S-22664'],
 [3, 2369125.41, 1839914.43, 'S-14301'],
 [4, 2086615.56, 1655432.88, 'S-4526'],
 [5, 1947974.56, 1573046.88, 'S-11386'],
 [6, 1870729.26, 1517452.08, 'S-4559'],
 [7, 1818051.78, 1483275.43, 'S-4548'],
 [8, 1784615.57, 1458446.76, 'S-4450'],
 [9, 1763676.88, 1446524.74, 'S-4245'],
 [10, 1744877.22, 1436474.64, 'S-4423'],
 [11, 1727362.7, 1423714.07, 'S-4702'],
 [12, 1713042.56, 1409796.25, 'S-23995'],
 [13, 1699946.41, 1400837.3, 'S-4304'],
 [14, 1690980.96, 1395921.41, 'S-4240'],
 [15, 1683104.5, 1389897.71, 'S-4829'],
 [16, 1675866.85, 1384704.65, 'S-13330'],
 [17, 1668961.29, 1377398.48, 'S-20496'],
 [18, 1663213.79, 1373305.72, 'S-4567'],
 [19, 1657905.84, 1370296.21, 'S-4103'],
 [20, 1652797.28, 1365572.71, 'S-16779'],
 [21, 1648201.85, 1360759.59, 'S-4761'],
 [22, 1643840.84, 1358704.37, 'S-4500'],
 [23, 1639921.16, 1357414.37, 'S-4080'],
 [24, 1636019.72, 1353604.63, 'S-4505'],
 [25, 1632377.64, 13

In [21]:
costs

,S-10655,S-10656,S-10659,S-10661,S-10662,S-10666,S-11252,S-11368,S-11369,S-11371,...,S-4988,S-4989,S-4990,S-4991,S-4992,S-4993,S-4994,S-4997,S-521,S-559
o,,,,,,,,,,,,,,,,,,,,,
337693613,16.029335,28.277057,18.033895,35.883725,24.454424,20.28337,8.422666,9.07407,6.185648,8.64407,...,8.88407,12.036702,7.118456,9.07407,7.791263,10.409684,9.866877,19.24337,9.435474,14.866527
337983850,16.029335,28.277057,18.033895,35.883725,24.454424,20.28337,8.422666,9.07407,6.185648,8.64407,...,8.88407,12.036702,7.118456,9.07407,7.791263,10.409684,9.866877,19.24337,9.435474,14.866527
338000532,16.029335,28.277057,18.033895,35.883725,24.454424,20.28337,8.422666,9.07407,6.185648,8.64407,...,8.88407,12.036702,7.118456,9.07407,7.791263,10.409684,9.866877,19.24337,9.435474,14.866527
338300849,16.029335,28.277057,18.033895,35.883725,24.454424,20.28337,8.422666,9.07407,6.185648,8.64407,...,8.88407,12.036702,7.118456,9.07407,7.791263,10.409684,9.866877,19.24337,9.435474,14.866527
338433340,16.029335,28.277057,18.033895,35.883725,24.454424,20.28337,8.422666,9.07407,6.185648,8.64407,...,8.88407,12.036702,7.118456,9.07407,7.791263,10.409684,9.866877,19.24337,9.435474,14.866527
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
342978659,16.029335,28.277057,18.033895,35.883725,24.454424,20.28337,8.422666,NaN,NaN,8.64407,...,8.88407,12.036702,7.118456,9.07407,7.791263,10.409684,9.866877,19.24337,9.435474,14.866527
344054824,16.029335,28.277057,18.033895,35.883725,24.454424,20.28337,8.422666,NaN,NaN,8.64407,...,8.88407,12.036702,7.118456,9.07407,7.791263,10.409684,9.866877,19.24337,9.435474,14.866527
349346311,16.029335,28.277057,NaN,35.883725,24.454424,20.28337,NaN,NaN,NaN,8.64407,...,8.88407,12.036702,NaN,9.07407,NaN,10.409684,9.866877,19.24337,NaN,14.866527


In [3]:
ul_bx = pd.read_csv(PATH + 'arka_boxes.csv')

df = pd.read_csv(PATH + 'data.csv')
es_df = df[['o', 'l', 'w', 'h', 'q', 'p']].copy()

[SLOPE, INTERCEPT, t, baseline_ba] = InferTranCost(df)

In [7]:
t

,o,Shipment Cost,Box Cost,BW,V_i,V_s
0,337479849,5.12,0.70,1.0,0,96.00000
1,337479946,6.43,3.66,3.0,0,306.28125
2,337480207,6.38,6.12,7.0,0,576.00000
3,337480314,7.72,9.94,8.0,0,1001.00000
4,337480497,15.95,3.66,3.0,0,306.28125
...,...,...,...,...,...,...
166627,362973597,7.16,2.04,5.0,0,576.00000
166628,362973638,6.98,7.04,21.0,0,2880.00000
166629,362973665,6.68,5.68,8.0,0,1001.00000
166630,362973678,7.16,2.04,5.0,0,576.00000


In [37]:
utilizations = []
print('size, transp, box, total, volume, pct unfit')
print('baseline')
#print(CURR_N_BOXES, round(h_c), round(h_b), round(h_t), round(t[['V_i', 'V_s']].max(axis = 1).sum() / 1728))
[pct_unfit,utilization] = Utilization(costs, pounds, baseline_boxes, ul_bx)
utilizations.append(utilization)
tot_cost = round(utilization['t'].sum())
box_cost = round((utilization['Total Utilization'] * utilization['c']).sum())
box_volume = round((utilization['Total Utilization'] * utilization['L'] * utilization['W'] * utilization['H']).sum() / 1728)
splits = round(utilization['Total Utilization'].sum() / costs.shape[0] - 1, 3)
print('original 4', tot_cost - box_cost, box_cost, tot_cost, box_volume, splits)
print('optimized')
#ord_with_cost = list(set(t['o']).intersection(set(costs.index)).intersection(set(df[df['n'] != 'none']['o'])))
for size in suite_sizes:
    [pct_unfit,utilization] = Utilization(costs, pounds, bao[size][2], ul_bx)
    utilizations.append(utilization)
    tot_cost = round(utilization['t'].sum())
    box_cost = round((utilization['Total Utilization'] * utilization['c']).sum())
    box_volume = round((utilization['Total Utilization'] * utilization['L'] * utilization['W'] * utilization['H']).sum() / 1728)
    splits = round(utilization['Total Utilization'].sum() / costs.shape[0] - 1, 3)
    print(size, tot_cost - box_cost, box_cost, tot_cost, box_volume, splits)


size, transp, box, total, volume, pct unfit
baseline
original 4 531583 56080 587663 8590 0.294
optimized
3 411709 31267 442976 9057 0.01
4 409238 30581 439819 8486 0.01
5 406985 30055 437040 7781 0.01
6 406984 29148 436132 7361 0.01
7 406428 29110 435538 7269 0.01
8 406428 28694 435122 7042 0.01


In [21]:
utilization

,Box Name,Total Utilization,Cumulative Utilization Pct,Dims,t,L,W,H,c
0,S-4050,19752,31.2%,5x5x5,150337.68,5.0,5.0,5.0,0.39
1,S-4060,15129,55.0%,6x4x4,115381.05,6.0,4.0,4.0,0.35
2,S-4062,11394,73.0%,6x6x6,102457.38,6.0,6.0,6.0,0.42
3,S-4362,7835,85.3%,9x6x6,82592.25,9.0,6.0,6.0,0.57
4,S-4080,2218,88.8%,8x6x4,21800.14,8.0,6.0,4.0,0.43
5,S-4153,2078,92.1%,15x12x10,28181.80,15.0,12.0,10.0,1.40
6,S-4121,2023,95.3%,12x8x6,27443.84,12.0,8.0,6.0,0.68
7,S-4624,1023,96.9%,15x11x7,10782.42,15.0,11.0,7.0,1.84
8,S-19870,506,97.7%,36x30x12,21763.06,36.0,30.0,12.0,8.81
9,S-4105,440,98.4%,10x10x10,7478.00,10.0,10.0,10.0,0.79


In [33]:
costs.shape[0]

62785

In [26]:
#ch = 
costs[baseline_boxes].reset_index().melt(id_vars = ['o'], var_name = 'n', value_name = 't').sort_values('t').groupby('o', as_index = False).first()
#round(ch[ch['t'].isna()].shape[0] / ch.shape[0], 2)

,o,n,t
0,L-6107618,S-4594,18.34
1,L-6107619,S-16700,7.92
2,L-6107620,S-4582,7.65
3,L-6107621,S-16700,7.92
4,L-6107622,S-4594,20.74
...,...,...,...
62780,L-6171113,S-4582,NaN
62781,L-6171114,S-4582,NaN
62782,L-6171115,S-4594,19.54
62783,L-6171116,S-16700,7.92


In [27]:
df[df['o'] == 'L-6171113']

,Unnamed: 0,o,l,w,h,p,q,s
35307,35307,L-6171113,4.250000,3.500000,2.750000,3.3,1,28LNOCC
50714,50714,L-6171113,14.566929,10.511811,6.496063,5.5,1,R-UTC-12


In [25]:
utilizations = []
print('size, transp, box, total, volume, pct unfit')
print('baseline')
#print(CURR_N_BOXES, round(h_c), round(h_b), round(h_t), round(t[['V_i', 'V_s']].max(axis = 1).sum() / 1728))
[pct_unfit,utilization] = Utilization(costs, pounds, baseline_boxes, ul_bx)
utilizations.append(utilization)
tot_cost = round(utilization['t'].sum())
box_cost = round((utilization['Total Utilization'] * utilization['c']).sum())
box_volume = round((utilization['Total Utilization'] * utilization['L'] * utilization['W'] * utilization['H']).sum() / 1728)
print('original 4', tot_cost - box_cost, box_cost, tot_cost, box_volume, pct_unfit)
print('optimized')
#ord_with_cost = list(set(t['o']).intersection(set(costs.index)).intersection(set(df[df['n'] != 'none']['o'])))
for size in suite_sizes:
    [pct_unfit,utilization] = Utilization(costs, pounds, bao[size][2], ul_bx)
    utilizations.append(utilization)
    tot_cost = round(utilization['t'].sum())
    box_cost = round((utilization['Total Utilization'] * utilization['c']).sum())
    box_volume = round((utilization['Total Utilization'] * utilization['L'] * utilization['W'] * utilization['H']).sum() / 1728)
    print(size, tot_cost - box_cost, box_cost, tot_cost, box_volume, pct_unfit)

size, transp, box, total, volume, pct unfit
baseline
original 4 549716 47630 597346 7257 0.0679
optimized
4 556254 43609 599863 20694 0.0
9 549646 37804 587450 17182 0.0
15 548023 36867 584890 15038 0.0


In [ ]:
#sb, first
bao = {20: (84963.847, 85017.66, ['S-4406', 'S-4187', 'S-4103', 'S-4143', 'S-4129', 'S-4175', 'S-4949', 'S-4397', 
                                  'S-4310', 'S-4153', 'S-4363', 'S-18369', 'S-4157', 'S-19827', 'S-4181', 'S-4350', 
                                  'S-12655', 'S-4162', 'S-4799', 'S-4206']), 
       23: (84398.453, 84412.85, ['S-4406', 'S-4187', 'S-4362', 'S-4103', 'S-4312', 'S-4143', 'S-4129', 'S-4175', 
                                  'S-4949', 'S-4397', 'S-4153', 'S-4363', 'S-16758', 'S-18369', 'S-4157', 'S-20497', 
                                  'S-19827', 'S-4181', 'S-4350', 'S-12655', 'S-4162', 'S-4799', 'S-4206']), 
       26: (83938.456, 83957.01, ['S-4406', 'S-4187', 'S-4362', 'S-4061', 'S-4103', 'S-4312', 'S-4143', 'S-4129', 
                                  'S-4175', 'S-4949', 'S-4397', 'S-4153', 'S-4363', 'S-16758', 'S-18369', 'S-4157', 
                                  'S-20497', 'S-19827', 'S-4181', 'S-4657', 'S-11386', 'S-4350', 'S-12655', 'S-4162', 'S-4799', 'S-4206'])}

In [21]:
actual_u = pd.DataFrame({'Package': {0: 'Box-10x8x3',
  1: 'Box-13x10x8',
  2: 'Box-6x5x4',
  3: 'Box-10x6x5',
  4: 'Box-17x13x7',
  5: 'Box-12x9x5',
  6: 'Box-14x12x3',
  7: 'Box-26x20x10',
  8: 'Box-18x16x8',
  9: 'Box-15x13x10',
  10: 'Box-15x11x4',
  11: 'Box-12x12x12',
  12: 'Box-20x20x20',
  13: 'Box-8x6x5',
  14: 'Box-12x6x4',
  15: 'Box-12x9x7',
  16: 'Box-28x3x3',
  17: 'Box-28x6x6',
  18: 'Box-6x6x2'},
 'OrderId': {0: 1266,
  1: 1127,
  2: 909,
  3: 831,
  4: 811,
  5: 808,
  6: 543,
  7: 435,
  8: 338,
  9: 286,
  10: 260,
  11: 100,
  12: 62,
  13: 10,
  14: 7,
  15: 4,
  16: 1,
  17: 1,
  18: 1}})
actual_u

,Package,OrderId
0,Box-10x8x3,1266
1,Box-13x10x8,1127
2,Box-6x5x4,909
3,Box-10x6x5,831
4,Box-17x13x7,811
5,Box-12x9x5,808
6,Box-14x12x3,543
7,Box-26x20x10,435
8,Box-18x16x8,338
9,Box-15x13x10,286


In [29]:
baseline_u = utilizations[0][['Box Name', 'Total Utilization']].merge(match[['n_x', 'n_y']], 
                                                         left_on='Box Name', right_on='n_y', how='outer').drop(columns = 'Box Name').fillna(0.)
baseline_u = baseline_u.merge(actual_u, left_on='n_x', right_on='Package', how='outer').drop(columns = 'Package').fillna(0.)
baseline_u = baseline_u.rename(columns = {'Total Utilization': '% sim', 'n_x': 'Name actual', 'n_y': 'Name sim', 'OrderId': '% act'})
baseline_u['% sim'] = np.round(baseline_u['% sim'] * 100 / baseline_u['% sim'].sum(),1)
baseline_u['% act'] = np.round(baseline_u['% act'] * 100 / baseline_u['% act'].sum(), 1)
baseline_u[['Name actual', 'Name sim', '% act', '% sim']]

,Name actual,Name sim,% act,% sim
0,Box-12x6x4,S-4127,0.1,20.0
1,Box-12x9x5,S-4487,10.4,11.8
2,Box-17x13x7,S-18388,10.4,7.6
3,Box-13x10x8,S-4838,14.4,7.1
4,Box-5x5x5,S-4050,0.0,6.5
5,Box-6x5x4,S-4512,11.7,5.9
6,Box-26x20x10,S-4507,5.6,5.6
7,Box-15x13x10,S-23974,3.7,5.2
8,Box-12x9x7,S-4488,0.1,5.0
9,Box-12x12x12,S-4125,1.3,4.9
